In [2]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.107.0
    Uninstalling openai-1.107.0:
      Successfully uninstalled openai-1.107.0


In [6]:
import openai
import pandas as pd

In [19]:
openai.api_key = 'Your-api'
# comment the below line if you use original OPENAI api
openai.api_base = 'https://api.avalai.ir/v1'

In [4]:
prompt_extraction = f'''
an answer to a multiple choice question is given to you, your task is
to extract the option that has been chosen,
Please provide the extracted answer in the given format,without any additional explanation
# Format
[answer] e.g. [B]
# Answer
{{answer}}
'''

In [16]:
def get_unlocal_AI_answer(prompt,m):
    response = openai.ChatCompletion.create(
        model=m,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature = 0.0
    )
    result = response['choices'][0]['message']['content'].strip().lower()
    return result

def get_unlocal_AI_answer(prompt,m):
    response = openai.ChatCompletion.create(
        model=m,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature = 0.0
    )
    result = response['choices'][0]['message']['content'].strip().lower()
    return result

def extract_choice(answer):
    extr = get_unlocal_AI_answer(prompt_extraction.format(answer=answer),'deepseek-chat')
    if extr == '[a]' or  extr == '[A]' or extr == 'a' or extr == 'A' or extr == '[الف]' or extr == 'الف' or extr == '[آ]' or extr == 'آ' or extr == '[ا]' or extr == 'ا':
       return 'A'
    elif extr == '[b]' or extr == '[B]' or extr == 'b' or extr == 'B' or extr == '[ب]' or extr == 'ب':
       return 'B'
    elif extr == '[c]' or extr == '[C]' or extr == 'c' or extr == 'C' or extr == '[ج]' or extr == 'ج':
       return 'C'
    elif extr == '[d]' or extr == '[D]' or extr == 'd' or extr == 'D' or extr == '[د]' or extr == 'د':
       return 'D'
    else:
       return 'invalid'
def append_record_to_excel(file_path,Question,question_choices,
                           correct_answer,model_prompt,AI_answer,AI_chosen_answer):
    new_record = {
        'Question': Question,
        'question_choices': question_choices,
        'correct_answer': correct_answer,
        'model_prompt': model_prompt,
        'AI_answer': AI_answer,
        'AI_chosen_answer':AI_chosen_answer
    }
    new_record_df = pd.DataFrame([new_record])
    try:
        existing_df = pd.read_excel(file_path)
        updated_df = pd.concat([existing_df, new_record_df], ignore_index=True)
    except FileNotFoundError:
        updated_df = new_record_df

    updated_df.to_excel(file_path, index=False)

In [21]:
df = pd.read_excel('a.xlsx')
for row in df.iterrows():
    AI_chosen_answer = extract_choice(row[1]['AI_answer'])
    Question = row[1]['Question']
    question_choices = row[1]['question_choices']
    correct_answer = row[1]['correct_answer']
    model_prompt = row[1]['model_prompt']
    AI_answer = row[1]['AI_answer']
    append_record_to_excel('fa_qwn_result.xlsx',Question,question_choices,
                           correct_answer,model_prompt,AI_answer,AI_chosen_answer)